# 06.11_All_h5_to_h5ad_Python

整合 H5 转 AnnData。

- 当前文件：`analysis/06_single_cell_analysis/06.11_All_h5_to_h5ad_Python.ipynb`
- 原始来源：`Codes/06.11_h5_to_h5ad.ipynb`（旧编号仅用于溯源）。
- 运行内核：**python**。
- 导入依赖：`os`, `pandas`, `scanpy`。
- 当前编号与流程见 `docs/workflow.md`、`docs/code_index.md`。
- 仅更新整理版导读；原分析单元格、参数和顺序保持不变。原始 cell 索引在本文件中加 1。


# h5 to h5ad

### 整合后的9 Species

In [ ]:
# python-code for converting h5 to h5ad
import pandas as pd
import scanpy as sc

fp_mat = "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/SingleCellIntegrated/Seurat_RPCA_to_Scanpy/sc_BasalMetazoa.matrix.data.h5"
fp_meta = "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/SingleCellIntegrated/Seurat_RPCA_to_Scanpy/sc_BasalMetazoa.metadata.csv"
adata = sc.read_10x_h5(fp_mat)
metadata = pd.read_csv(fp_meta, index_col=0)
# add meta-data
for c in metadata.columns:
    if metadata[c].dtype == object:
        adata.obs[c] = metadata[c].astype(str)  # 对象类型通常是字符串，明确转换以避免问题
    else:
        adata.obs[c] = metadata[c]  # 直接赋值

print(adata)

In [ ]:
import os

pca_path = "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/SingleCellIntegrated/Seurat_RPCA_to_Scanpy/sc_BasalMetazoa.reduction.pca.csv"
umap_path = "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/SingleCellIntegrated/Seurat_RPCA_to_Scanpy/sc_BasalMetazoa.reduction.umap.csv"
# tsne_path = "/share/home/zhangze/zz/NeuralOrigin/Data/03.SingleCellProcessing/scMatrix/AU_scMatrix_CycloneSeq/sc_Auco.reduction.tsne.csv"

if os.path.exists(pca_path):
    pca_df = pd.read_csv(pca_path, index_col=0)
    # Align and assign
    adata.obsm['X_pca'] = pca_df.loc[adata.obs_names, :].values

if os.path.exists(umap_path):
    umap_df = pd.read_csv(umap_path, index_col=0)
    adata.obsm['X_umap'] = umap_df.loc[adata.obs_names, :].values

# if os.path.exists(tsne_path):
#     tsne_df = pd.read_csv(tsne_path, index_col=0)
#     adata.obsm['X_tsne'] = tsne_df.loc[adata.obs_names, :].values

print(adata)

In [ ]:
adata.obs

In [ ]:
adata.var

In [ ]:
adata.X.toarray()

In [ ]:
adata

In [ ]:
# 保留原始 count 数据
adata.layers["counts"] = adata.X.copy()

In [ ]:
# 标准化和 log 转换
# sc.pp.normalize_total(adata, target_sum=1e4)
# sc.pp.log1p(adata)

# 高度变异基因选择（如果需要）
sc.pp.highly_variable_genes(adata, n_top_genes=1000)

print(adata)

In [ ]:
# 保存原始数据
adata.raw = adata.copy()

In [ ]:
adata.X.toarray()

In [ ]:
sc.pl.umap(adata, color=['species', 'Phylum', 'BroadType'])

In [ ]:
# 保存adata数据
raw_9species_path = "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/SingleCellIntegrated/Seurat_RPCA_to_Scanpy/sc_BasalMetazoa.normalized.h5ad"
adata.write(raw_9species_path)

In [ ]:
# 导出基因id
adata.var_names.to_series().to_csv('/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/SingleCellIntegrated/Seurat_RPCA_to_Scanpy/sc_BasalMetazoa.genes.txt', index=False, header=False)